# Temperature Forecasting: Model Comparison
## Actual vs LightGBM vs GRU vs Hybrid Ensemble

### Cell 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

import os
os.makedirs('data/processed/plots', exist_ok=True)

print("✓ Libraries imported successfully")

### Cell 2: Load Prediction Data

In [ ]:
print("Loading prediction files...")

lgbm_val = pd.read_csv('data/processed/lgbm_val_preds.csv')
lgbm_test = pd.read_csv('data/processed/lgbm_test_preds.csv')

gru_val = pd.read_csv('data/processed/gru_val_preds.csv')
gru_test = pd.read_csv('data/processed/gru_test_preds.csv')

val_df = lgbm_val.merge(gru_val[['Date', 'City', 'gru_pred']], on=['Date', 'City'])
test_df = lgbm_test.merge(gru_test[['Date', 'City', 'gru_pred']], on=['Date', 'City'])

val_df['Date'] = pd.to_datetime(val_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])

print(f"✓ Validation samples: {len(val_df)}")
print(f"✓ Test samples: {len(test_df)}")

### Cell 3: Generate Hybrid Predictions (Ridge Stacking)

In [ ]:
val_df['pred_diff'] = val_df['lgbm_pred'] - val_df['gru_pred']
test_df['pred_diff'] = test_df['lgbm_pred'] - test_df['gru_pred']

features = ['lgbm_pred', 'gru_pred', 'pred_diff']
X_train = val_df[features].values
y_train = val_df['y_true'].values
X_test = test_df[features].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=0.01)
ridge.fit(X_train_scaled, y_train)

test_df['hybrid_pred'] = ridge.predict(X_test_scaled)

print("✓ Hybrid predictions generated")

### Cell 4: Calculate Performance Metrics

In [ ]:
models = {
    'LightGBM': test_df['lgbm_pred'],
    'GRU': test_df['gru_pred'],
    'Hybrid': test_df['hybrid_pred']
}

for name, preds in models.items():
    rmse = np.sqrt(mean_squared_error(test_df['y_true'], preds))
    mae = mean_absolute_error(test_df['y_true'], preds)
    r2 = r2_score(test_df['y_true'], preds)
    print(f"{name}: RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

### Cell 5: Time Series Plot

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(test_df['y_true'].values[:200], label='Actual')
plt.plot(test_df['lgbm_pred'].values[:200], label='LightGBM')
plt.plot(test_df['gru_pred'].values[:200], label='GRU')
plt.plot(test_df['hybrid_pred'].values[:200], label='Hybrid')
plt.legend()
plt.title("Model Comparison (First 200 Samples)")
plt.show()

### Cell 6: Error Distribution

In [ ]:
lgbm_error = test_df['y_true'] - test_df['lgbm_pred']
gru_error = test_df['y_true'] - test_df['gru_pred']
hybrid_error = test_df['y_true'] - test_df['hybrid_pred']

plt.hist(lgbm_error, bins=50, alpha=0.5, label='LGBM')
plt.hist(gru_error, bins=50, alpha=0.5, label='GRU')
plt.hist(hybrid_error, bins=50, alpha=0.5, label='Hybrid')
plt.legend()
plt.title("Error Distribution")
plt.show()

### Cell 7: Save Outputs

In [ ]:
test_df.to_csv('data/processed/final_hybrid_predictions.csv', index=False)
print("✓ Saved predictions")